In [ ]:
from dotenv import load_dotenv
from openai import OpenAI
from wikifin_rag.rag_helper import RAGBase
from wikifin_rag.embedder import Embedder
from wikifin_rag.evaluation_utils import llm_structured_retry, map_progress, calc_total_price
from concurrent.futures import ThreadPoolExecutor
import json
from wikifin_rag.config import PROJECT_ROOT
import pandas as pd
from tqdm.auto import tqdm

In [2]:
embedder = Embedder()

In [3]:
load_dotenv(override=True)
openai_client = OpenAI()

In [4]:
assistant = RAGBase(
    embedder=embedder,
    llm_client=openai_client
)

In [5]:
db_client = assistant.db_client

In [6]:
db_client.open_connection()

db_client.cur.execute(
    """
    SELECT
        c.document_id || '_' || c.chunk_id AS id,
        d.title,
        d.section,
        c.content
    FROM chunks c
    JOIN documents d
    ON c.document_id = d.id
    WHERE language = 'nl'
    LIMIT 100;
    """
)
documents = db_client.cur.fetchall()

db_client.close_connection()

In [7]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [8]:
data_gen_instructions = """
You emulate either a university student or a young professional.
Formulate 3 questions this student/professional might ask based on an article excerpt. The excerpt
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [ ]:
def generate_document_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [ ]:
def generate_corpus_ground_truth(documents, dest="data"):
    with ThreadPoolExecutor(max_workers=6) as pool:
        results = map_progress(pool, documents, generate_document_ground_truth)

    ground_truth = []
    usages = []

    for records, usage in results:
        ground_truth.extend(records)
        usages.append(usage)

    total_cost = calc_total_price(usages)

    # save the generated data to a CSV file
    df_ground_truth = pd.DataFrame(ground_truth)
    dest = PROJECT_ROOT / dest
    dest.mkdir(parents=True, exist_ok=True)
    df_ground_truth.to_csv(dest / "ground_truth.csv", index=False)

    return ground_truth, total_cost

In [ ]:
df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [15]:
def vector_search(query):
    return assistant.search(query, num_results=5)

In [23]:
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])
    relevance = [int(d["id"] == doc_id) for d in results]
    return relevance

In [24]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [25]:
ground_truth_sample = ground_truth[:15]
relevance_total = compute_relevance_total(ground_truth_sample, vector_search)

  0%|          | 0/15 [00:00<?, ?it/s]

In [ ]:
relevance_total

In [27]:
openai_client.close()